# Aviation Accidents Analysis - Data Cleaning

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023.

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**

In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# The raw file contains non-UTF-8 characters, so latin-1 encoding is used
df = pd.read_csv('AviationData.csv', encoding='latin-1', low_memory=False)

print(df.shape)
df.head()

(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
# Datatypes and missing values per column
summary = pd.DataFrame({'dtype': df.dtypes.astype(str), 'n_missing': df.isna().sum(), 'pct_missing': (df.isna().mean() * 100).round(1)})
summary

,dtype,n_missing,pct_missing
Event.Id,str,0,0.0
Investigation.Type,str,0,0.0
Accident.Number,str,0,0.0
Event.Date,str,0,0.0
Location,str,52,0.1
Country,str,226,0.3
Latitude,str,54507,61.3
Longitude,str,54516,61.3
Airport.Code,str,38757,43.6
Airport.Name,str,36185,40.7


In [4]:
# Summary statistics of the numeric columns
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


**Observations**
- 88,889 events and 31 columns, with data from 1948 to 2022.
- Several columns are mostly empty (Latitude, Longitude, Airport.Code, Airport.Name, FAR.Description, Schedule, Air.carrier). These are dropped at the end.
- `Aircraft.Category` is missing for roughly 64% of events, so it must be imputed before we can restrict the analysis to airplanes.
- The four injury count columns (`Total.Fatal.Injuries`, `Total.Serious.Injuries`, `Total.Minor.Injuries`, `Total.Uninjured`) each have thousands of NaNs.
- Text columns (Weather.Condition, Engine.Type, Purpose.of.flight, Make) have inconsistent labels (e.g. `UNK` / `Unk` / `Unknown`).

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

**Filtering assumptions**
1. *Max aircraft lifetime of 40 years*: keep events from 1983 onwards.
2. *Professional builds only*: keep `Amateur.Built == 'No'` (the 102 events with no value are dropped since we cannot confirm they are professional builds).
3. *Airplanes only*: the dataset also contains helicopters, gliders, balloons, etc. `Aircraft.Category` is missing for most rows, so we impute it (see below) before filtering.

In [5]:
# Parse dates and filter on year and professional builds
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df['Year'] = df['Event.Date'].dt.year

df = df[(df['Year'] >= 1983) & (df['Amateur.Built'] == 'No')].copy()
print(df.shape)

(76960, 32)


In [6]:
# Standardize Make/Model text first so that imputation using make/model works consistently
df['Make'] = df['Make'].str.upper().str.strip()
df['Model'] = df['Model'].str.upper().str.strip()

df['Aircraft.Category'].value_counts(dropna=False)

Aircraft.Category
NaN                  51555
Airplane             21447
Helicopter            3022
Glider                 434
Balloon                199
Weight-Shift           139
Powered Parachute       83
Gyrocraft               31
Ultralight              18
Unknown                 11
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
Name: count, dtype: int64

**Imputing `Aircraft.Category`**

A make/model that is labelled `Airplane` in other rows is almost certainly an airplane in the rows where the category is missing. The imputation is done in two steps:
1. Fill with the most common labelled category of the same Make + Model.
2. For the remaining rows, fill with the most common labelled category of the same Make *if* that make is at least 95% a single category (with at least 20 labelled rows), e.g. Cessna is >99% airplanes.

Rows that still have no category after this are dropped, since we cannot tell if they are airplanes.

In [7]:
labelled = df[df['Aircraft.Category'].notna()]

# Step 1: Make + Model lookup
model_map = labelled.groupby(['Make', 'Model'])['Aircraft.Category'].agg(lambda s: s.value_counts().index[0])
keys = pd.MultiIndex.from_frame(df[['Make', 'Model']])
model_fill = pd.Series(model_map.reindex(keys).values, index=df.index)

# Step 2: Make lookup (only for makes that are consistently one category)
make_stats = labelled.groupby('Make')['Aircraft.Category'].agg(
    top=lambda s: s.value_counts().index[0],
    share=lambda s: s.value_counts(normalize=True).iloc[0],
    n='size')
make_map = make_stats[(make_stats['share'] >= 0.95) & (make_stats['n'] >= 20)]['top']
make_fill = df['Make'].map(make_map)

print('Missing before:', df['Aircraft.Category'].isna().sum())
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(model_fill).fillna(make_fill)
print('Missing after:', df['Aircraft.Category'].isna().sum())

# Keep airplanes only
df = df[df['Aircraft.Category'] == 'Airplane'].copy()
print(df.shape)

Missing before: 51555
Missing after: 2982
(64938, 32)


In [8]:
# Only keep standard accident investigations that have full details? Check the investigation type
df['Investigation.Type'].value_counts()

Investigation.Type
Accident    61765
Incident     3173
Name: count, dtype: int64

`Incident` records are minor events that are still safety relevant (they contain the aircraft damage and injury fields we need), so both types are kept.

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Assumptions*
- The total number of people on board is estimated as `Total.Fatal.Injuries + Total.Serious.Injuries + Total.Minor.Injuries + Total.Uninjured`.
- If an injury count is missing but at least one of the four columns is reported, the missing count is assumed to be 0. (Examples: events with `Injury.Severity` = `Non-Fatal` and a missing fatal count have no fatalities.)
- Events where all four columns are missing, or where the estimated total is 0, are dropped because no fraction can be computed.
- New derived columns: `Total.Passengers` and `Fatal.Serious.Fraction = (Fatal + Serious) / Total.Passengers`.

In [9]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

print('Rows with all four injury columns missing:', df[injury_cols].isna().all(axis=1).sum())
df = df[~df[injury_cols].isna().all(axis=1)].copy()

# Missing counts are assumed to be 0 when at least one count was reported
df[injury_cols] = df[injury_cols].fillna(0)

df['Total.Passengers'] = df[injury_cols].sum(axis=1)
print('Rows with zero people on board:', (df['Total.Passengers'] == 0).sum())
df = df[df['Total.Passengers'] > 0].copy()

df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

print(df.shape)
df[['Total.Passengers', 'Fatal.Serious.Fraction']].describe()

Rows with all four injury columns missing: 145
Rows with zero people on board: 969
(63824, 34)


,Total.Passengers,Fatal.Serious.Fraction
count,63824.000000,63824.000000
mean,7.214355,0.268406
std,31.391257,0.427681
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,3.000000,0.600000
max,699.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [10]:
df['Aircraft.damage'].value_counts(dropna=False)

Aircraft.damage
Substantial    47156
Destroyed      12872
NaN             1873
Minor           1849
Unknown           74
Name: count, dtype: int64

*Cleaning tasks*
- `Aircraft.damage` has NaN and `Unknown` values. We cannot tell if those aircraft were destroyed, so these rows are dropped.
- Derived column: `Destroyed` = 1 if `Aircraft.damage == 'Destroyed'`, otherwise 0 (Substantial or Minor).

In [11]:
df = df[df['Aircraft.damage'].isin(['Destroyed', 'Substantial', 'Minor'])].copy()
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

print(df.shape)
df['Destroyed'].value_counts(normalize=True)

(61877, 35)


Destroyed
0    0.791974
1    0.208026
Name: proportion, dtype: float64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Cleaning tasks**
1. Drop rows with a missing Make.
2. Standardize case and whitespace (done earlier: upper case and stripped).
3. Merge variants of the same manufacturer, e.g. `CESSNA AIRCRAFT CO`, `HAWKER BEECHCRAFT CORP`, `AIRBUS INDUSTRIE`, `CIRRUS DESIGN CORP.`, `DEHAVILLAND`, `GATES LEARJET`.
4. Keep Makes with at least 50 events so that comparisons are statistically meaningful.

In [12]:
print('Unique makes before:', df['Make'].nunique())
df = df.dropna(subset=['Make']).copy()

def clean_make(name):
    # Merge common name variants of the same manufacturer
    rules = [
        ('CESSNA', 'CESSNA'),
        ('NEW PIPER', 'PIPER'),
        ('PIPER AIRCRAFT', 'PIPER'),
        ('BEECH', 'BEECH'),           # Beech, Beechcraft, Hawker Beechcraft
        ('HAWKER BEECH', 'BEECH'),
        ('BOEING STEARMAN', 'BOEING STEARMAN'),
        ('BOEING', 'BOEING'),
        ('THE BOEING', 'BOEING'),
        ('AIRBUS', 'AIRBUS'),
        ('MCDONNELL DOUGLAS', 'MCDONNELL DOUGLAS'),
        ('DEHAVILLAND', 'DE HAVILLAND'),
        ('DE HAVILLAND', 'DE HAVILLAND'),
        ('EMBRAER', 'EMBRAER'),
        ('BOMBARDIER', 'BOMBARDIER'),
        ('CIRRUS', 'CIRRUS'),
        ('MOONEY', 'MOONEY'),
        ('GRUMMAN-SCHWEIZER', 'GRUMMAN'),
        ('GRUMMAN ACFT', 'GRUMMAN'),
        ('GRUMMAN AMERICAN', 'GRUMMAN'),
        ('GRUMMAN', 'GRUMMAN'),
        ('SAAB', 'SAAB'),
        ('AIR TRACTOR', 'AIR TRACTOR'),
        ('ROCKWELL', 'ROCKWELL'),
        ('AMERICAN CHAMPION', 'CHAMPION'),
        ('CHAMPION', 'CHAMPION'),
        ('AYRES', 'AYRES'),
        ('GATES LEARJET', 'LEARJET'),
        ('LEARJET', 'LEARJET'),
        ('GULFSTREAM', 'GULFSTREAM'),
        ('DIAMOND', 'DIAMOND'),
        ('PILATUS', 'PILATUS'),
        ('FAIRCHILD SWEARINGEN', 'SWEARINGEN'),
        ('SWEARINGEN', 'SWEARINGEN'),
        ('SHORT BRO', 'SHORT BROTHERS'),
        ('AEROSTAR', 'AEROSTAR'),
        ('SMITH, TED AEROSTAR', 'AEROSTAR'),
        ('EVEKTOR', 'EVEKTOR'),
        ('LIBERTY AEROSPACE', 'LIBERTY AEROSPACE'),
    ]
    for prefix, standard in rules:
        if name.startswith(prefix):
            return standard
    if name.startswith('PIPER'):
        return 'PIPER'
    return name

df['Make'] = df['Make'].apply(clean_make)
print('Unique makes after merging variants:', df['Make'].nunique())

# Keep Makes with a reasonable number of events
make_counts = df['Make'].value_counts()
keep_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(keep_makes)].copy()

print('Makes kept:', len(keep_makes))
print(df.shape)
df['Make'].value_counts().head(20)

Unique makes before: 1051
Unique makes after merging variants: 920
Makes kept: 50
(59241, 35)


Make
CESSNA            25445
PIPER             14001
BEECH              5069
GRUMMAN            1505
MOONEY             1304
BOEING             1304
BELLANCA            973
AIR TRACTOR         890
AERONCA             604
CHAMPION            588
MAULE               569
DE HAVILLAND        461
CIRRUS              448
STINSON             421
AERO COMMANDER      405
ROCKWELL            405
LUSCOMBE            386
NORTH AMERICAN      366
TAYLORCRAFT         364
AYRES               276
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [13]:
print('Missing models:', df['Model'].isna().sum())
df = df.dropna(subset=['Model']).copy()

# Tidy up whitespace inside model names
df['Model'] = df['Model'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Are model labels shared between makes?
models_per_label = df.groupby('Model')['Make'].nunique()
print('Model labels used by more than one make:', (models_per_label > 1).sum())
models_per_label[models_per_label > 1].sort_values(ascending=False).head(10)

Missing models: 13


Model labels used by more than one make: 181


Model
500       5
G-164A    4
112       3
400       3
200       3
35A       3
500-S     3
500B      3
500S      3
402       3
Name: Make, dtype: int64

Model labels such as `737`, `A320` or `172` are not unique identifiers on their own (some labels are shared across makes, e.g. numeric names like `100` or `200`). A derived column `Make.Model` = Make + Model is created as the unique plane type identifier.

In [14]:
df['Make.Model'] = df['Make'] + ' ' + df['Model']

print('Unique plane types:', df['Make.Model'].nunique())
df['Make.Model'].value_counts().head(15)

Unique plane types: 4424


Make.Model
CESSNA 152         2211
CESSNA 172         1614
CESSNA 172N        1081
PIPER PA-28-140     857
CESSNA 172M         755
CESSNA 150          742
CESSNA 172P         661
CESSNA 182          608
CESSNA 180          593
PIPER PA-18-150     553
CESSNA 150M         550
PIPER PA-18         548
PIPER PA-28-180     545
PIPER PA-28-161     523
PIPER PA-28-181     504
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Cleaning tasks identified**
- **Weather.Condition**: `UNK`, `Unk`, `Unknown` and NaN all mean unknown conditions. Standardize to `Unknown`. Valid categories are VMC (visual) and IMC (instrument) conditions.
- **Engine.Type**: `UNK`, `NONE`, `LR`, NaN and `Unknown` are merged into `Unknown`. `Geared Turbofan` is merged into `Turbo Fan`.
- **Number.of.Engines**: an airplane cannot have 0 engines, so 0 is treated as missing (NaN). The remaining NaNs are left as NaN.
- **Purpose.of.flight**: merge `Public Aircraft - Federal/Local/State`, `PUBS`, `PUBL` into `Public Aircraft`; `Air Race/show` into `Air Race show`; `ASHO` into `Air Race show`; NaN into `Unknown`.
- **Broad.phase.of.flight**: NaN is set to `Unknown`.

In [15]:
# Weather
df['Weather.Condition'] = df['Weather.Condition'].str.upper().replace({'UNK': 'Unknown', 'UNKNOWN': 'Unknown'}).fillna('Unknown')

# Engine type
df['Engine.Type'] = df['Engine.Type'].replace({'UNK': 'Unknown', 'NONE': 'Unknown', 'LR': 'Unknown', 'Geared Turbofan': 'Turbo Fan'}).fillna('Unknown')

# Number of engines: 0 engines is not valid for an airplane
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)

# Purpose of flight
df['Purpose.of.flight'] = (df['Purpose.of.flight']
    .replace({'Public Aircraft - Federal': 'Public Aircraft',
              'Public Aircraft - Local': 'Public Aircraft',
              'Public Aircraft - State': 'Public Aircraft',
              'PUBS': 'Public Aircraft', 'PUBL': 'Public Aircraft',
              'Air Race/show': 'Air Race show', 'ASHO': 'Air Race show'})
    .fillna('Unknown'))

# Phase of flight
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Unknown')

for col in ['Weather.Condition', 'Engine.Type', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[col].value_counts(dropna=False).head(12), '\n')

Weather.Condition
VMC        52025
IMC         4725
Unknown     2478
Name: count, dtype: int64 

Engine.Type
Reciprocating    51568
Unknown           3491
Turbo Prop        2622
Turbo Fan         1203
Turbo Jet          323
Turbo Shaft         21
Name: count, dtype: int64 

Number.of.Engines
1.0    48414
2.0     7914
NaN     2411
4.0      250
3.0      239
Name: count, dtype: int64 

Purpose.of.flight
Personal               34762
Instructional           8039
Unknown                 6452
Aerial Application      3399
Business                2922
Positioning             1003
Ferry                    550
Other Work Use           517
Public Aircraft          386
Aerial Observation       376
Executive/corporate      344
Skydiving                171
Name: count, dtype: int64 

Broad.phase.of.flight
Unknown        16460
Landing        11858
Takeoff         8834
Cruise          7082
Maneuvering     4607
Approach        4527
Taxi            1475
Climb           1420
Descent         1249
Go-around

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [16]:
pct_missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
pct_missing[pct_missing > 0]

Schedule               88.1
Air.carrier            83.2
FAR.Description        70.5
Longitude              62.7
Latitude               62.7
Airport.Code           39.0
Airport.Name           36.1
Publication.Date       17.3
Report.Status           5.2
Number.of.Engines       4.1
Registration.Number     1.1
Country                 0.1
dtype: float64

In [17]:
# Drop columns that are more than 40% empty, plus identifiers that are not useful for the analysis
too_many_nans = pct_missing[pct_missing > 40].index.tolist()
print('Dropping:', too_many_nans)

df = df.drop(columns=too_many_nans + ['Accident.Number', 'Registration.Number', 'Publication.Date', 'Report.Status'], errors='ignore')

print(df.shape)
df.info()

Dropping: ['Schedule', 'Air.carrier', 'FAR.Description', 'Longitude', 'Latitude']
(59228, 27)


<class 'pandas.DataFrame'>
Index: 59228 entries, 3601 to 88886
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                59228 non-null  str           
 1   Investigation.Type      59228 non-null  str           
 2   Event.Date              59228 non-null  datetime64[us]
 3   Location                59200 non-null  str           
 4   Country                 59140 non-null  str           
 5   Airport.Code            36147 non-null  str           
 6   Airport.Name            37838 non-null  str           
 7   Injury.Severity         59228 non-null  str           
 8   Aircraft.damage         59228 non-null  str           
 9   Aircraft.Category       59228 non-null  str           
 10  Make                    59228 non-null  str           
 11  Model                   59228 non-null  str           
 12  Amateur.Built           59228 non-null  str           
 13 

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [18]:
df.to_csv('cleaned_aviation_data.csv', index=False)
print(df.shape)

(59228, 27)
